# Text-Fabric to Neo4j

Use this notebook to convert a Text-Fabric dataset into a Neo4j graph.

- Nodes become `(:TFNode {tf_id, otype, ...features})`
- Edge-features become `[:TF_EDGE {name, ...props}]`

Set paths and credentials in the config cell, then run all cells.

In [1]:
%pip install text-fabric neo4j python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

# Make sure src/ is importable when notebook is started from repo root or notebooks/
repo_root = Path.cwd()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from tf2neo4j import TFExportConfig, export_text_fabric_to_neo4j

In [5]:
# --- Configure your dataset + Neo4j connection ---
# Text-Fabric dataset folder (required)
TF_LOCATIONS = r"C:\Users\tonyj\text-fabric-data\github\etcbc\bhsa-min\tf\2021"

# Optional TF module name/list. Keep as None for plain local data.
TF_MODULES = None

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

# None means: use all available TF node/edge features
NODE_FEATURES = None
EDGE_FEATURES = None

# Tuning
BATCH_SIZE = 2000
CLEAR_DATABASE = False

In [6]:
config = TFExportConfig(
    tf_locations=TF_LOCATIONS,
    tf_modules=TF_MODULES,
    neo4j_uri=NEO4J_URI,
    neo4j_user=NEO4J_USER,
    neo4j_password=NEO4J_PASSWORD,
    neo4j_database=NEO4J_DATABASE,
    node_features=NODE_FEATURES,
    edge_features=EDGE_FEATURES,
    batch_size=BATCH_SIZE,
    clear_database=CLEAR_DATABASE,
)

stats = export_text_fabric_to_neo4j(config)
print(f"Export complete: {stats.node_count} nodes, {stats.relationship_count} relationships")

Unable to retrieve routing information


ServiceUnavailable: Unable to retrieve routing information

In [ ]:
from neo4j import GraphDatabase

with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as driver:
    with driver.session(database=NEO4J_DATABASE) as session:
        n = session.run("MATCH (n:TFNode) RETURN count(n) AS c").single()["c"]
        r = session.run("MATCH ()-[r:TF_EDGE]->() RETURN count(r) AS c").single()["c"]

print({"nodes": n, "relationships": r})